# Laboratorio 6 -- Análisis de redes sociales en YouTube

Universidad del Valle de Guatemala. CC3084 Data Science, Semestre II 2026.

Fernando Rueda -- 23748. Fernando Hernández -- 23645.

Estudiamos la estructura de participación en YouTube a partir de dos conjuntos,
uno de videos y canales y otro de comentarios. Este avance cubre la carga y la
integración de los datos, un diagnóstico de calidad, la limpieza y el
preprocesamiento del texto, el análisis exploratorio y la construcción de la red
bipartita autor-video. Los datos no permiten saber quién respondió a quién, así
que el conteo de respuestas no se interpreta como una relación entre usuarios.

## Ejercicio 1. Carga, comprensión e integración de los datos

### 1.1 Carga de los archivos

Cargamos los dos archivos con pandas y revisamos sus dimensiones.

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SALIDAS = ROOT / "salidas"
SALIDAS.mkdir(exist_ok=True)

videos = pd.read_csv(ROOT / "data" / "youtube_videos.csv")
comentarios = pd.read_csv(ROOT / "data" / "youtube_comments.csv")
print("videos:", videos.shape, "| comentarios:", comentarios.shape)

videos: (293, 20) | comentarios: (406, 17)


### 1.2 Unidad de observación, llave primaria y variables relevantes

En el archivo de videos cada fila es un video de YouTube, la llave primaria es
video_id y las variables más relevantes son el canal que lo publicó, con channel_id
como identificador estable y channel_name como nombre visible, el título, la
categoría, el número de visualizaciones y la consulta con la que se recolectó. En
el archivo de comentarios cada fila es un comentario principal, la llave primaria es
comment_id y las variables clave son el texto del comentario, el autor, con
author_channel_id como identificador y author_name como nombre visible, el video al
que pertenece a través de video_id, y los conteos de me gusta y de respuestas.

In [2]:
def resumen(df, nombre):
    print(f"== {nombre} ==")
    print("dimensiones:", df.shape)
    print("columnas:", list(df.columns))

resumen(videos, "videos")
print()
resumen(comentarios, "comentarios")
print("\n¿video_id es único en videos?", videos["video_id"].is_unique)
print("¿comment_id es único en comentarios?", comentarios["comment_id"].is_unique)

== videos ==
dimensiones: (293, 20)
columnas: ['video_id', 'title', 'channel_name', 'channel_id', 'source_query', 'source_group', 'dataset_sources', 'channel_handle', 'published_time', 'view_count_text', 'description_snippet', 'video_url', 'query_hits', 'keywords', 'description', 'view_count', 'publish_date', 'upload_date', 'category', 'owner_handle']

== comentarios ==
dimensiones: (406, 17)
columnas: ['video_id', 'comment_id', 'video_title', 'channel_name', 'channel_id', 'author_name', 'author_channel_id', 'text', 'source_query', 'source_group', 'dataset_sources', 'author_handle', 'published_text', 'like_count_text', 'reply_count', 'is_pinned', 'viewer_rating']

¿video_id es único en videos? True
¿comment_id es único en comentarios? True


### 1.3 Relación entre canal, video, autor, comentario, categoría y consulta

Un canal, identificado por channel_id, publica videos, y cada video pertenece a una
categoría de YouTube y fue recolectado mediante una consulta de búsqueda descrita por
source_query y source_group. Cada comentario se publica en un video, relación que se
establece con video_id, y lo escribe un autor identificado por author_channel_id. Es
importante notar que el canal del video y el autor del comentario son entidades
distintas, el primero es quien subió el video y el segundo quien comentó, y que un
mismo autor puede comentar en varios videos. El conteo de respuestas de un comentario
no dice quién respondió, así que no define una relación entre autores.

### 1.4 Integración por video_id

Unimos los comentarios con la información de su video a través de video_id y
reportamos cuántos comentarios pudieron asociarse a un video del conjunto.

In [3]:
ids_video = set(videos["video_id"])
con_video = comentarios["video_id"].isin(ids_video)
print("comentarios totales:", len(comentarios))
print("comentarios asociados a un video del conjunto:", int(con_video.sum()))
print("comentarios sin video en el conjunto:", int((~con_video).sum()))
print("videos con al menos un comentario:", comentarios["video_id"].nunique())

integrado = comentarios.merge(
    videos, on="video_id", how="left", suffixes=("_com", "_vid"))
print("\ntabla integrada:", integrado.shape)

comentarios totales: 406
comentarios asociados a un video del conjunto: 406
comentarios sin video en el conjunto: 0
videos con al menos un comentario: 19

tabla integrada: (406, 36)
